# Lab 01: API Testing — Test Clocks

**Duration**: ~15 minutes  
**Prerequisites**: Completed `02_sandbox_and_test_cards.ipynb`

## Learning Objectives

By the end of this notebook, you will:
- Understand why test clocks exist and when to use them
- Create and advance test clocks via the API
- Simulate a subscription trial, first payment, renewal, and payment failure

---

In [12]:
!pip install stripe --quiet

import os
import stripe
import time
from datetime import datetime

try:
    from google.colab import userdata
    stripe.api_key = userdata.get('STRIPE_SECRET_KEY')
    print("Loaded API key from Colab Secrets.")
except Exception:
    import getpass
    stripe.api_key = getpass.getpass("Paste your Stripe test secret key (sk_test_...): ")

try:
    account = stripe.Account.retrieve()
    mode = 'Sandbox' if 'test' in stripe.api_key else 'LIVE — BE CAREFUL!'
    print(f"Connected to account: {account.id} | Mode: {mode}")
except stripe.error.AuthenticationError:
    print("ERROR: Invalid API key.")

Loaded API key from Colab Secrets.
Connected to account: acct_1RnL4mBMxfUzotEq | Mode: Sandbox


---

## Why Test Clocks?

Subscriptions operate over time. Without test clocks, testing a monthly subscription requires waiting a month.

```
Without test clocks:
  Day 1   → Create subscription with 7-day trial
  Day 8   → Trial ends, first payment charged
  Day 38  → First renewal
  ...wait...

With test clocks:
  Minute 1 → Create subscription with 7-day trial
  Minute 2 → Advance clock 8 days  → trial ends, first payment
  Minute 3 → Advance clock 30 days → renewal
  Minute 4 → Change card to declining, advance → failed renewal
```

### Lifecycle

```
Create Clock (set frozen_time)
        ↓
Create Customer (attached to clock)
        ↓
Create Subscription
        ↓
Advance Clock  →  billing events fire
        ↓
Repeat or delete
```

### Limitations

- Advance at most **2 billing intervals** per call
- Up to **3 customers** per test clock
- Only available in **sandboxes** (`sk_test_` keys)

---

## Exercise 1: Create a Test Clock

In [13]:
# frozen_time is a Unix timestamp — start from right now
frozen_time = int(time.time())

test_clock = stripe.test_helpers.TestClock.create(
    frozen_time=frozen_time,
    name="Colab Workshop — Subscription Test"
)

CLOCK_ID = test_clock.id

print(f"Test Clock ID: {CLOCK_ID}")
print(f"Frozen Time:   {datetime.fromtimestamp(test_clock.frozen_time)}")
print(f"Status:        {test_clock.status}")

Test Clock ID: clock_1TMUffBMxfUzotEqdab0ai73
Frozen Time:   2026-04-15 14:44:02
Status:        ready


**Dashboard**: [Test Clocks](https://dashboard.stripe.com/test/test-clocks) — you should see your new clock.

---

## Exercise 2: Create Customer, Product, Price, and Subscription

In [14]:
# Customer MUST be created with test_clock= to participate in simulated time
customer = stripe.Customer.create(
    email="test-clock-demo@example.com",
    name="Test Clock Demo User",
    test_clock=CLOCK_ID,
    payment_method="pm_card_visa",
    invoice_settings={"default_payment_method": "pm_card_visa"}
)

CUSTOMER_ID = customer.id
print(f"Customer ID: {CUSTOMER_ID}")
print(f"Test Clock:  {customer.test_clock}")

Customer ID: cus_ULB1RbpedcSHDC
Test Clock:  clock_1TMUffBMxfUzotEqdab0ai73


In [15]:
# Create a product and monthly price
product = stripe.Product.create(
    name="Workshop Pro Plan",
    description="Monthly subscription for test clock demo"
)

price = stripe.Price.create(
    product=product.id,
    unit_amount=2900,   # $29.00
    currency="usd",
    recurring={"interval": "month"}
)

PRICE_ID = price.id
print(f"Product: {product.name} ({product.id})")
print(f"Price:   ${price.unit_amount / 100:.2f}/month ({PRICE_ID})")

Product: Workshop Pro Plan (prod_ULB16Ryl1P8Ktg)
Price:   $29.00/month (price_1TMUfgBMxfUzotEqcVxmtKxE)


In [16]:
# Create subscription with a 7-day free trial
subscription = stripe.Subscription.create(
    customer=CUSTOMER_ID,
    items=[{"price": PRICE_ID}],
    trial_period_days=7
)

SUBSCRIPTION_ID = subscription.id
print(f"Subscription ID: {SUBSCRIPTION_ID}")
print(f"Status:          {subscription.status}")
print(f"Trial ends:      {datetime.fromtimestamp(subscription.trial_end)}")
# Expected status: trialing

Subscription ID: sub_1TMUfhBMxfUzotEquteJyVh1
Status:          trialing
Trial ends:      2026-04-22 14:44:02


---

## Exercise 3: Advance Time

The table below shows what happens at each advance:

| Advance to | Events fired |
|------------|--------------|
| Day 5 (2 days before trial end) | `customer.subscription.trial_will_end` |
| Day 8 (past trial) | `customer.subscription.updated` (active), `invoice.paid` |
| Day 38 (next billing cycle) | `invoice.upcoming`, `invoice.paid` |

In [17]:
def advance_clock(clock_id: str, days: int):
    """Advance a test clock by `days` days and wait for it to finish."""
    clock = stripe.test_helpers.TestClock.retrieve(clock_id)
    current_time = clock.frozen_time
    new_time = current_time + days * 24 * 60 * 60

    print(f"Advancing {days} day(s):  {datetime.fromtimestamp(current_time)} → {datetime.fromtimestamp(new_time)}")

    stripe.test_helpers.TestClock.advance(clock_id, frozen_time=new_time)

    # Poll until the clock finishes advancing
    for _ in range(30):
        clock = stripe.test_helpers.TestClock.retrieve(clock_id)
        if clock.status != "advancing":
            break
        time.sleep(1)

    print(f"Done. Clock is now: {datetime.fromtimestamp(clock.frozen_time)} (status: {clock.status})")
    return clock

print("advance_clock() defined.")

advance_clock() defined.


In [18]:
# Advance 8 days — past the trial end — to trigger first payment
print("=" * 60)
print("Advancing past trial period...")
print("=" * 60)

advance_clock(CLOCK_ID, days=8)

subscription = stripe.Subscription.retrieve(SUBSCRIPTION_ID)
print(f"\nSubscription status: {subscription.status}")
# Expected: active (no longer trialing)

Advancing past trial period...
Advancing 8 day(s):  2026-04-15 14:44:02 → 2026-04-23 14:44:02
Done. Clock is now: 2026-04-23 14:44:02 (status: ready)

Subscription status: active


In [19]:
# Check invoices — there should be a paid invoice for $29.00
invoices = stripe.Invoice.list(customer=CUSTOMER_ID, limit=5)

print("Invoices after trial ended:")
for inv in invoices.data:
    print(f"  {inv.id} | ${inv.amount_paid / 100:.2f} | status: {inv.status} | {datetime.fromtimestamp(inv.created)}")

Invoices after trial ended:
  in_1TMUfmBMxfUzotEqC2pM22qS | $29.00 | status: paid | 2026-04-22 14:44:02
  in_1TMUfhBMxfUzotEqB32pYlzf | $0.00 | status: paid | 2026-04-15 14:44:02


In [20]:
# Advance 30 more days to trigger the monthly renewal
print("=" * 60)
print("Advancing to next billing cycle (renewal)...")
print("=" * 60)

advance_clock(CLOCK_ID, days=30)

invoices = stripe.Invoice.list(customer=CUSTOMER_ID, limit=5)
print("\nInvoices after renewal:")
for inv in invoices.data:
    print(f"  {inv.id} | ${inv.amount_paid / 100:.2f} | status: {inv.status}")

Advancing to next billing cycle (renewal)...
Advancing 30 day(s):  2026-04-23 14:44:02 → 2026-05-23 14:44:02
Done. Clock is now: 2026-05-23 14:44:02 (status: ready)

Invoices after renewal:
  in_1TMUg1BMxfUzotEqVQhfCNa0 | $29.00 | status: paid
  in_1TMUfmBMxfUzotEqC2pM22qS | $29.00 | status: paid
  in_1TMUfhBMxfUzotEqB32pYlzf | $0.00 | status: paid


---

## Exercise 4: Simulate a Payment Failure on Renewal

Update the customer's default payment method to one that will decline, then advance to the next billing cycle.

In [21]:
# Attach a card that will always fail
failing_pm = stripe.PaymentMethod.attach(
    "pm_card_chargeCustomerFail",
    customer=CUSTOMER_ID,
)

stripe.Customer.modify(
    CUSTOMER_ID,
    invoice_settings={"default_payment_method": failing_pm.id}
)

print("Customer's default payment method updated to a failing card.")
print("The next renewal attempt will fail.")

Customer's default payment method updated to a failing card.
The next renewal attempt will fail.


In [22]:
# Advance 30 more days — renewal will fail
print("=" * 60)
print("Advancing to trigger failed renewal...")
print("=" * 60)

advance_clock(CLOCK_ID, days=30)

subscription = stripe.Subscription.retrieve(SUBSCRIPTION_ID)
print(f"\nSubscription status: {subscription.status}")

invoices = stripe.Invoice.list(customer=CUSTOMER_ID, limit=1)
if invoices.data:
    latest = invoices.data[0]
    print(f"\nLatest Invoice:")
    print(f"  ID:             {latest.id}")
    print(f"  Amount due:     ${latest.amount_due / 100:.2f}")
    print(f"  Status:         {latest.status}")
    print(f"  Attempt count:  {latest.attempt_count}")
# Expected: invoice status = open, subscription status = past_due

Advancing to trigger failed renewal...
Advancing 30 day(s):  2026-05-23 14:44:02 → 2026-06-22 14:44:02
Done. Clock is now: 2026-06-22 14:44:02 (status: ready)

Subscription status: active

Latest Invoice:
  ID:             in_1TMUgFBMxfUzotEq74rM0DCd
  Amount due:     $29.00
  Status:         draft
  Attempt count:  0


**Dashboard**: [Invoices](https://dashboard.stripe.com/test/invoices) — you should see the failed invoice marked as `open`.

---

## Cleanup

In [23]:
print("Cleaning up test resources...")

try:
    stripe.Subscription.cancel(SUBSCRIPTION_ID)
    print(f"  Cancelled subscription: {SUBSCRIPTION_ID}")
except Exception as e:
    print(f"  Could not cancel subscription: {e}")

try:
    stripe.test_helpers.TestClock.delete(CLOCK_ID)
    print(f"  Deleted test clock: {CLOCK_ID}")
except Exception as e:
    print(f"  Could not delete clock: {e}")

print("Done.")

Cleaning up test resources...
  Cancelled subscription: sub_1TMUfhBMxfUzotEquteJyVh1
  Deleted test clock: clock_1TMUffBMxfUzotEqdab0ai73
Done.


---

## Summary

- **Test clocks** simulate time passage — compress months of billing into minutes
- Attach a customer to a test clock at creation time via `test_clock=`
- **Advancing** the clock fires real billing events (`invoice.paid`, `customer.subscription.updated`, etc.)
- Switch the default payment method to a failing card to test dunning and `past_due` flows
- Always **delete test clocks** when done to keep your sandbox tidy



## Next Steps

Open `04_webhook_fundamentals.ipynb` to start Lab 02: Webhook Integration.